# Day 8 — Segmentation & Explainable Catalog CV

**RetailMind AI | Phase 2 — Weekly Assessment 03**

This increment uses OpenCV GrabCut to create pseudo-masks, trains a U-Net-style TensorFlow/Keras segmenter, composites clean catalog images, adds Grad-CAM, and packages a unified interface.

**Dataset limitation:** Fashion Product Images Small does not contain pixel-level segmentation labels. GrabCut masks are pseudo-labels, not ground truth. Validation IoU measures agreement with GrabCut, not real-world segmentation accuracy.

## 1. Imports and project paths
Update `PROJECT_ROOT` if your repository location differs.

In [ ]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
PROJECT_ROOT=Path.cwd()
if PROJECT_ROOT.name=="notebooks": PROJECT_ROOT=PROJECT_ROOT.parent
DATA_DIR=PROJECT_ROOT/"data"
MODEL_DIR=PROJECT_ROOT/"models"
ARTIFACT_DIR=PROJECT_ROOT/"artifacts"/"day8"
MASK_DIR=DATA_DIR/"processed"/"segmentation_pseudo_masks"
for p in [MODEL_DIR,ARTIFACT_DIR,MASK_DIR]: p.mkdir(parents=True,exist_ok=True)
IMAGE_SIZE=128
print("Project:",PROJECT_ROOT,"TensorFlow:",tf.__version__,"OpenCV:",cv2.__version__)

## 2. Locate the Kaggle dataset
Extract `paramaggarwal/fashion-product-images-small` under `data/raw/fashion-product-images-small`, or edit `DATASET_DIR`.

In [ ]:
candidates=[DATA_DIR/"raw"/"fashion-product-images-small",DATA_DIR/"raw"/"fashion_product_images_small",PROJECT_ROOT/"fashion-product-images-small"]
DATASET_DIR=next((p for p in candidates if (p/"styles.csv").exists()),None)
if DATASET_DIR is None: raise FileNotFoundError("Set DATASET_DIR to the extracted folder containing styles.csv and images/.")
IMAGE_DIR=DATASET_DIR/"images"
styles=pd.read_csv(DATASET_DIR/"styles.csv",on_bad_lines="skip")
styles["id"]=pd.to_numeric(styles["id"],errors="coerce")
styles=styles.dropna(subset=["id"]).copy()
styles["id"]=styles["id"].astype(int)
styles["image_path"]=styles["id"].astype(str)+".jpg"
styles["image_path"]=styles["image_path"].map(lambda x:IMAGE_DIR/x)
styles=styles[styles.image_path.map(Path.exists)].reset_index(drop=True)
print("Usable images:",len(styles))
styles.head()

## 3. Generate GrabCut pseudo-masks
GrabCut is initialized with a central rectangle, which is a reasonable baseline for centered catalog photos. Inspect outputs: it can fail on off-center products and complex backgrounds.

In [ ]:
def grabcut_mask(image_path,size=IMAGE_SIZE,iterations=5):
    bgr=cv2.imread(str(image_path))
    if bgr is None: raise ValueError(f"Cannot read {image_path}")
    h,w=bgr.shape[:2]
    mx,my=max(1,int(w*.06)),max(1,int(h*.06))
    rect=(mx,my,max(2,w-2*mx),max(2,h-2*my))
    mask=np.zeros((h,w),np.uint8)
    bg=np.zeros((1,65),np.float64); fg=np.zeros((1,65),np.float64)
    try:
        cv2.grabCut(bgr,mask,rect,bg,fg,iterations,cv2.GC_INIT_WITH_RECT)
        mask=np.where((mask==cv2.GC_FGD)|(mask==cv2.GC_PR_FGD),1,0).astype(np.uint8)
        mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,np.ones((3,3),np.uint8))
        mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((5,5),np.uint8))
    except cv2.error:
        mask=np.zeros((h,w),np.uint8)
        mask[int(.08*h):int(.92*h),int(.08*w):int(.92*w)]=1
    return cv2.resize(mask,(size,size),interpolation=cv2.INTER_NEAREST)

MAX_IMAGES=min(2500,len(styles))
sample=styles.sample(MAX_IMAGES,random_state=SEED).reset_index(drop=True)
def cache_mask(row):
    p=MASK_DIR/f"{row.id}.png"
    if not p.exists(): cv2.imwrite(str(p),grabcut_mask(row.image_path)*255)
    return str(p)
sample["mask_path"]=sample.apply(cache_mask,axis=1)
print("Pseudo-masks ready:",len(sample))

## 4. Inspect pseudo-mask quality
Review examples before training. Remove or correct bad masks if products are clipped or background is included.

In [ ]:
def inspect_masks(frame,n=5):
    chosen=frame.sample(min(n,len(frame)),random_state=SEED)
    fig,axs=plt.subplots(len(chosen),3,figsize=(10,3*len(chosen)))
    if len(chosen)==1: axs=np.expand_dims(axs,0)
    for ax,(_,r) in zip(axs,chosen.iterrows()):
        rgb=cv2.cvtColor(cv2.imread(str(r.image_path)),cv2.COLOR_BGR2RGB)
        rgb=cv2.resize(rgb,(IMAGE_SIZE,IMAGE_SIZE))
        mask=cv2.imread(r.mask_path,cv2.IMREAD_GRAYSCALE)
        ax[0].imshow(rgb); ax[0].set_title("Original"); ax[0].axis("off")
        ax[1].imshow(mask,cmap="gray"); ax[1].set_title("Pseudo-mask"); ax[1].axis("off")
        ax[2].imshow(rgb*(mask[...,None]>127)); ax[2].set_title("Foreground"); ax[2].axis("off")
    plt.tight_layout()
inspect_masks(sample)

## 5. TensorFlow input pipeline

In [ ]:
shuffled=sample.sample(frac=1,random_state=SEED).reset_index(drop=True)
split_at=max(1,int(len(shuffled)*.85))
train_df=shuffled.iloc[:split_at].copy()
val_df=shuffled.iloc[split_at:].copy()
def load_pair(image_path,mask_path):
    x=tf.io.decode_jpeg(tf.io.read_file(image_path),channels=3)
    x=tf.image.resize(x,[IMAGE_SIZE,IMAGE_SIZE])/255.
    y=tf.io.decode_png(tf.io.read_file(mask_path),channels=1)
    y=tf.image.resize(y,[IMAGE_SIZE,IMAGE_SIZE],method="nearest")
    return tf.cast(x,tf.float32),tf.cast(y>127,tf.float32)
def make_ds(frame,training=False):
    ds=tf.data.Dataset.from_tensor_slices((frame.image_path.astype(str).values,frame.mask_path.values))
    ds=ds.map(load_pair,num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds=ds.shuffle(min(len(frame),1000),seed=SEED)
        ds=ds.map(lambda x,y:(tf.image.random_flip_left_right(x),tf.image.random_flip_left_right(y)))
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)
train_ds=make_ds(train_df,True); val_ds=make_ds(val_df)

## 6. Build and train a U-Net-style segmenter
The model learns to imitate GrabCut pseudo-labels. BCE + Dice loss helps with foreground/background imbalance.

In [ ]:
def dice_loss(y_true,y_pred,smooth=1.):
    yt=tf.reshape(tf.cast(y_true,tf.float32),[-1]); yp=tf.reshape(tf.cast(y_pred,tf.float32),[-1])
    inter=tf.reduce_sum(yt*yp)
    return 1-(2*inter+smooth)/(tf.reduce_sum(yt)+tf.reduce_sum(yp)+smooth)
def seg_loss(y_true,y_pred):
    return tf.reduce_mean(keras.losses.binary_crossentropy(y_true,y_pred))+dice_loss(y_true,y_pred)
def block(x,f):
    x=layers.Conv2D(f,3,padding="same",activation="relu")(x)
    x=layers.BatchNormalization()(x)
    x=layers.Conv2D(f,3,padding="same",activation="relu")(x)
    return layers.BatchNormalization()(x)
def build_unet():
    i=keras.Input((IMAGE_SIZE,IMAGE_SIZE,3))
    c1=block(i,32); p1=layers.MaxPooling2D()(c1)
    c2=block(p1,64); p2=layers.MaxPooling2D()(c2)
    c3=block(p2,128); p3=layers.MaxPooling2D()(c3)
    b=block(p3,256)
    d3=block(layers.Concatenate()([layers.UpSampling2D()(b),c3]),128)
    d2=block(layers.Concatenate()([layers.UpSampling2D()(d3),c2]),64)
    d1=block(layers.Concatenate()([layers.UpSampling2D()(d2),c1]),32)
    return keras.Model(i,layers.Conv2D(1,1,activation="sigmoid")(d1),name="retailmind_unet")
seg_model=build_unet()
seg_model.compile(optimizer=keras.optimizers.Adam(1e-3),loss=seg_loss,
 metrics=[keras.metrics.BinaryIoU(target_class_ids=[1],threshold=.5,name="foreground_iou")])
seg_model.summary()
callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss",patience=4,restore_best_weights=True),
 keras.callbacks.ModelCheckpoint(MODEL_DIR/"day8_unet_segmentation.keras",save_best_only=True)]
history=seg_model.fit(train_ds,validation_data=val_ds,epochs=20,callbacks=callbacks)

## 7. Segment and composite onto a clean background

In [ ]:
def segment_rgb(rgb,model=seg_model,threshold=.5):
    h,w=rgb.shape[:2]
    x=cv2.resize(rgb,(IMAGE_SIZE,IMAGE_SIZE)).astype(np.float32)/255.
    soft=model.predict(x[None,...],verbose=0)[0,...,0]
    mask=cv2.resize((soft>=threshold).astype(np.uint8)*255,(w,h),interpolation=cv2.INTER_NEAREST)
    alpha=cv2.GaussianBlur(mask.astype(np.float32)/255.,(5,5),0)
    return mask,alpha,soft
def remove_background(image_path,output_path=None,background=(255,255,255),threshold=.5):
    bgr=cv2.imread(str(image_path))
    if bgr is None: raise ValueError(f"Cannot read {image_path}")
    rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    mask,alpha,soft=segment_rgb(rgb,threshold=threshold)
    bg=np.full_like(rgb,background,dtype=np.uint8)
    result=(rgb.astype(np.float32)*alpha[...,None]+bg.astype(np.float32)*(1-alpha[...,None])).clip(0,255).astype(np.uint8)
    if output_path:
        Path(output_path).parent.mkdir(parents=True,exist_ok=True)
        cv2.imwrite(str(output_path),cv2.cvtColor(result,cv2.COLOR_RGB2BGR))
    return result,mask,soft
example=sample.iloc[0].image_path
clean,mask,_=remove_background(example,ARTIFACT_DIR/"sample_clean_catalog.jpg")
plt.figure(figsize=(5,5)); plt.imshow(clean); plt.axis("off"); plt.title("Clean catalog image"); plt.show()

## 8. Grad-CAM for the Day 6/7 classifier
Set the model path to your existing `.keras` classifier. Grad-CAM requires a differentiable classifier and a convolutional layer. If the last convolution is nested inside a backbone, set `CONV_LAYER_NAME` explicitly and adapt the layer lookup.

In [ ]:
CLASSIFIER_PATH=MODEL_DIR/"day7_resnet50_finetuned.keras"
classifier=keras.models.load_model(CLASSIFIER_PATH,compile=False) if CLASSIFIER_PATH.exists() else None
if classifier is None: print("Set CLASSIFIER_PATH to your Day 6/7 classifier artifact.")
else: classifier.summary()

In [ ]:
def gradcam(image_path,model,conv_layer_name,class_index=None):
    bgr=cv2.imread(str(image_path))
    if bgr is None: raise ValueError("Cannot read image")
    rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    size=int(model.input_shape[1])
    x=cv2.resize(rgb,(size,size)).astype(np.float32)/255.
    gm=keras.Model(model.inputs,[model.get_layer(conv_layer_name).output,model.output])
    with tf.GradientTape() as tape:
        conv_out,preds=gm(x[None,...],training=False)
        idx=tf.argmax(preds[0]) if class_index is None else class_index
        score=preds[:,idx]
    grads=tape.gradient(score,conv_out)
    weights=tf.reduce_mean(grads,axis=(0,1,2))
    heat=tf.reduce_sum(conv_out[0]*weights,axis=-1)
    heat=tf.maximum(heat,0)/(tf.reduce_max(heat)+keras.backend.epsilon())
    heat=cv2.resize(heat.numpy(),(rgb.shape[1],rgb.shape[0]))
    color=cv2.cvtColor(cv2.applyColorMap(np.uint8(255*heat),cv2.COLORMAP_JET),cv2.COLOR_BGR2RGB)
    return rgb,cv2.addWeighted(rgb,.6,color,.4,0),preds.numpy()

if classifier is not None:
    CONV_LAYER_NAME="conv5_block3_3_conv" # Typical ResNet50 final conv; verify with classifier.summary().
    original,overlay,preds=gradcam(example,classifier,CONV_LAYER_NAME)
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1); plt.imshow(original); plt.title("Input"); plt.axis("off")
    plt.subplot(1,2,2); plt.imshow(overlay); plt.title("Grad-CAM"); plt.axis("off")
    plt.show()
    print("Predicted class index:",int(np.argmax(preds[0])))

## 9. Package classification + detection + segmentation + explainability
The detector is injected as a callable adapter to your Day 7 detector. This keeps this increment limited to TensorFlow/Keras and OpenCV; it does not reimplement or add a new detection framework.

In [ ]:
package_dir=PROJECT_ROOT/"src"/"retailmind_cv"
package_dir.mkdir(parents=True,exist_ok=True)
module_code = r'''
from pathlib import Path
import cv2, numpy as np, tensorflow as tf
from tensorflow import keras

class RetailMindCV:
    def __init__(self,classifier=None,segmenter=None,detector=None,class_names=None,
                 classifier_size=224,segmentation_size=128,conv_layer_name=None,
                 classifier_preprocess=None):
        self.classifier=classifier; self.segmenter=segmenter; self.detector=detector
        self.class_names=class_names or []; self.classifier_size=classifier_size
        self.segmentation_size=segmentation_size; self.conv_layer_name=conv_layer_name
        self.classifier_preprocess=classifier_preprocess
    def _read(self,path):
        bgr=cv2.imread(str(path))
        if bgr is None: raise ValueError(f"Cannot read {path}")
        return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    def classify(self,path):
        if self.classifier is None: raise RuntimeError("No classifier attached")
        x=cv2.resize(self._read(path),(self.classifier_size,self.classifier_size)).astype(np.float32)
        x=self.classifier_preprocess(x) if self.classifier_preprocess else x/255.
        p=self.classifier.predict(x[None,...],verbose=0)[0]; idx=int(np.argmax(p))
        return {"class_index":idx,"label":self.class_names[idx] if idx<len(self.class_names) else str(idx),
                "confidence":float(p[idx]),"probabilities":p.tolist()}
    def detect(self,path):
        return self.detector(str(path)) if self.detector else {"available":False,"message":"Attach Day 7 detector adapter"}
    def segment(self,path,threshold=.5,output_path=None,background=(255,255,255)):
        if self.segmenter is None: raise RuntimeError("No segmenter attached")
        rgb=self._read(path); h,w=rgb.shape[:2]
        x=cv2.resize(rgb,(self.segmentation_size,self.segmentation_size)).astype(np.float32)/255.
        soft=self.segmenter.predict(x[None,...],verbose=0)[0,...,0]
        mask=cv2.resize((soft>=threshold).astype(np.uint8)*255,(w,h),interpolation=cv2.INTER_NEAREST)
        alpha=cv2.GaussianBlur(mask.astype(np.float32)/255.,(5,5),0)
        bg=np.full_like(rgb,background,dtype=np.uint8)
        out=(rgb*alpha[...,None]+bg*(1-alpha[...,None])).clip(0,255).astype(np.uint8)
        if output_path:
            Path(output_path).parent.mkdir(parents=True,exist_ok=True)
            cv2.imwrite(str(output_path),cv2.cvtColor(out,cv2.COLOR_RGB2BGR))
        return {"mask":mask,"composite":out}
    def explain(self,path,class_index=None):
        if self.classifier is None or not self.conv_layer_name: raise RuntimeError("Attach classifier and conv_layer_name")
        rgb=self._read(path)
        x=cv2.resize(rgb,(self.classifier_size,self.classifier_size)).astype(np.float32)
        x=self.classifier_preprocess(x) if self.classifier_preprocess else x/255.
        conv=self.classifier.get_layer(self.conv_layer_name)
        gm=keras.Model(self.classifier.inputs,[conv.output,self.classifier.output])
        with tf.GradientTape() as tape:
            co,p=gm(x[None,...],training=False); idx=tf.argmax(p[0]) if class_index is None else class_index
            score=p[:,idx]
        grads=tape.gradient(score,co); weights=tf.reduce_mean(grads,axis=(0,1,2))
        heat=tf.reduce_sum(co[0]*weights,axis=-1); heat=tf.maximum(heat,0)/(tf.reduce_max(heat)+keras.backend.epsilon())
        heat=cv2.resize(heat.numpy(),(rgb.shape[1],rgb.shape[0]))
        color=cv2.cvtColor(cv2.applyColorMap(np.uint8(255*heat),cv2.COLORMAP_JET),cv2.COLOR_BGR2RGB)
        return {"heatmap":heat,"overlay":cv2.addWeighted(rgb,.6,color,.4,0)}
'''
(package_dir/"__init__.py").write_text(module_code,encoding="utf-8")
print("Created",package_dir/"__init__.py")

## 10. Connect Day 7 detection and test the unified interface
Implement `day7_detector_adapter(path)` around your existing Day 7 detector. Return a list of detection dictionaries containing class, confidence, and bounding box values.

In [ ]:
# Example contract:
# def day7_detector_adapter(path):
#     results = your_existing_day7_detector(path)
#     return [{"class_id": 0, "confidence": 0.93, "box_xyxy": [x1,y1,x2,y2]}]

day7_detector_adapter=None
from src.retailmind_cv import RetailMindCV
saved_seg=keras.models.load_model(MODEL_DIR/"day8_unet_segmentation.keras",compile=False)
cv_module=RetailMindCV(classifier=classifier,segmenter=saved_seg,detector=day7_detector_adapter,
    classifier_size=int(classifier.input_shape[1]) if classifier is not None else 224,
    segmentation_size=IMAGE_SIZE,
    conv_layer_name=CONV_LAYER_NAME if classifier is not None else None)
test_result=cv_module.segment(example,output_path=ARTIFACT_DIR/"module_clean_output.jpg")
print("Segmentation output:",test_result["composite"].shape)
print("Detection:",cv_module.detect(example))
if classifier is not None: print("Classification:",cv_module.classify(example))

## 11. Export audit metrics and update README
Remember: pseudo-mask IoU is only a proxy for agreement with GrabCut. It is not a ground-truth segmentation score.

In [ ]:
metrics=saved_seg.evaluate(val_ds,return_dict=True,verbose=1)
with open(ARTIFACT_DIR/"pseudo_mask_validation_metrics.json","w") as f:
    json.dump({k:float(v) for k,v in metrics.items()},f,indent=2)
audit=[]
for _,row in val_df.head(30).iterrows():
    out=ARTIFACT_DIR/f"clean_{row.id}.jpg"
    clean,mask,_=remove_background(row.image_path,out)
    audit.append({"image_id":int(row.id),"source":str(row.image_path),"output":str(out),
                  "foreground_fraction":float(np.mean(mask>0))})
pd.DataFrame(audit).to_csv(ARTIFACT_DIR/"segmentation_audit.csv",index=False)
readme=PROJECT_ROOT/"README.md"
section='''
\n## Day 8 — Explainable Catalog CV
- OpenCV GrabCut pseudo-mask generation + Keras U-Net-style segmentation.
- Background removal and compositing on a uniform catalog background.
- Grad-CAM overlays for compatible Day 6/7 Keras classifiers.
- Unified interface: `src/retailmind_cv/` (classification, Day 7 detector adapter, segmentation, explainability).
- Outputs: `artifacts/day8/` (clean images, audit CSV, pseudo-mask metrics).

**Limitation:** The source dataset has no pixel-level masks. The U-Net is trained against GrabCut pseudo-labels; validation IoU measures agreement with those labels, not ground-truth segmentation quality.
'''
old=readme.read_text(encoding="utf-8") if readme.exists() else "# RetailMind AI\n"
if "## Day 8 — Explainable Catalog CV" not in old: readme.write_text(old.rstrip()+"\n"+section,encoding="utf-8")
print("Audit saved; README updated.")

## 12. Completion checklist and Git

- [ ] Confirm dataset path and inspect pseudo-masks.
- [ ] Train and save `models/day8_unet_segmentation.keras`.
- [ ] Generate clean-background catalog images.
- [ ] Load Day 6/7 classifier and generate Grad-CAM overlays.
- [ ] Connect the Day 7 detector adapter.
- [ ] Test `src/retailmind_cv`.
- [ ] Review README and document pseudo-label limitations.

Suggested commit:
```bash
git add src/retailmind_cv README.md
git commit -m "Add explainable catalog CV segmentation module"
```

Avoid committing the entire Kaggle dataset or large generated mask cache unless your repository tracks data artifacts. Use `.gitignore` or Git LFS as appropriate.